In [27]:
import gym

from acme import specs
import acme.wrappers as wrappers


environment = gym.make('LunarLanderContinuous-v2')
environment = wrappers.GymWrapper(environment)  # Convert to dm_env interface.

# Make sure the environment outputs single-precision floats.
environment = wrappers.SinglePrecisionWrapper(environment) 

environment_spec = specs.make_environment_spec(environment)

print(environment_spec)

EnvironmentSpec(observations=BoundedArray(shape=(8,), dtype=dtype('float32'), name='observation', minimum=[-inf -inf -inf -inf -inf -inf -inf -inf], maximum=[inf inf inf inf inf inf inf inf]), actions=BoundedArray(shape=(2,), dtype=dtype('float32'), name='action', minimum=[-1. -1.], maximum=[1. 1.]), rewards=Array(shape=(), dtype=dtype('float32'), name='reward'), discounts=BoundedArray(shape=(), dtype=dtype('float32'), name='discount', minimum=0.0, maximum=1.0))


In [87]:
import sonnet as snt
from acme.tf import networks
import numpy as np
import tensorflow as tf
from acme.tf import utils as tf2_utils

# network layer sizes
policy_layer_sizes = (256, 128)
critic_layer_sizes = (256, 128)

# Get total number of action dimensions from action spec.
action_spec = environment_spec.actions
action_size = np.prod(action_spec.shape, dtype=int)
print(action_size)

policy_network = snt.Sequential([
    tf2_utils.batch_concat,
    networks.LayerNormMLP(layer_sizes=(256, 128, action_size), activate_final=False),
    networks.TanhToSpec(spec=environment_spec.actions)
])

# The multiplexer concatenates the (maybe transformed) observations/actions.
# critic_network = networks.CriticMultiplexer(
#       critic_network=networks.LayerNormMLP((256, 128, 1)))

critic_network = snt.Sequential([
      # The multiplexer concatenates the observations/actions.
      networks.CriticMultiplexer(),
      networks.LayerNormMLP((256, 128, 1), activate_final=False),
  ])

2


In [85]:
from acme.agents.tf import ddpg
from acme.utils import loggers
from acme import environment_loop

# Create a logger for the agent and environment loop.
agent_logger = loggers.TerminalLogger(label='agent', time_delta=10.)
env_loop_logger = loggers.TerminalLogger(label='env_loop', time_delta=10.)

# Create the DDPG agent.
agent = ddpg.DDPG(environment_spec=environment_spec, policy_network=policy_network, critic_network=critic_network, logger=agent_logger, checkpoint=False)

# Create an loop connecting this agent to the environment created above.
env_loop = environment_loop.EnvironmentLoop(environment, agent, logger=env_loop_logger)

In [86]:
# Run a `num_episodes` training episodes.
# Rerun this cell until the agent has learned the given task.
env_loop.run(num_episodes=1)

ValueError: Unable to CreateItem to table priority_table because Append was called with a tensor signature inconsistent with table signature.  Saw a tensor at timestep offset 0 in (flattened) tensor location 2 with dtype int32 and shape [] but expected a tensor of dtype float and shape compatible with [].  (Flattened) table signature: 0: Tensor<name: '0', dtype: float, shape: [8]>, 1: Tensor<name: '1', dtype: float, shape: [2]>, 2: Tensor<name: '2', dtype: float, shape: []>, 3: Tensor<name: '3', dtype: float, shape: []>, 4: Tensor<name: '4', dtype: float, shape: [8]>